In [4]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import gymnasium as gym
from IPython.display import Video

# Reinforce with GAE

Instead of using monte carlo estimates to compute the returns, we instead use the generalized advantage estimation function. This is basically TD lambda but with advantages.

We define a recursive method that recurses back from the final reward at the final timestep, computes the advantage there and then multiplies it by lambda\*delta, then goes to the timestep before that, computes the generalized advantage est. there by computing

advantage*t = delta + gamma * lambda \* advantage_t+1

advantage*t-1 = delta + gamma * lambda \* advantage_t

advantage*t-2 = delta + gamma * lambda \* advantage_t-1

And so on, as you walk further down the reversed trajectory, you compute the n-step advantage iteratively


In [5]:
# initialize our policy network
class PolicyNetwork(nn.Module): 

    def __init__(self, input_features=8, output_features=4, hidden_features=128):
        super(PolicyNetwork, self).__init__()

        self.fc1 = nn.Linear(input_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, output_features)

    def forward(self, x):

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        pi = F.softmax(x, dim=-1)

        return pi

# initialize our policy network
class ValueNetwork(nn.Module): 

    def __init__(self, input_features=8, hidden_features=128):
        super().__init__()

        self.fc1 = nn.Linear(input_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, 1)

    def forward(self, x):

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [23]:
# Compute advantages for a trajectory
# A_t+1 = delta_t + gamma * lambda * A_t
def compute_gae(rewards, values, gamma, lam): 

    advantages = []
    returns = []

    advantage = 0
    next_value = 0

    for reward, value in zip(reversed(rewards), reversed(values)): 

        delta = reward + gamma*next_value - value # the TD error of advantage with 1 step look ahead
        advantage = delta + gamma * lam * advantage # recursive formula that gives us advantage for every step iteratively, this advantage is A_t^GAE

        g_t = advantage + value

        next_value = value # the value from this step (t+1) becomes the value for the next (t)

        advantages.insert(0, advantage) # insert advantage at the beginning so you return with the correct order 
        returns.insert(0, g_t)

    advantages = torch.tensor(advantages, dtype=torch.float32, device=values.device)
    returns = torch.tensor(returns, dtype=torch.float32, device=values.device)

    return advantages, returns

In [ ]:
# Training - where we run the game, calculate returns, calculate loss - entropy, then step the model
def train(
    env,
    entropy_weight=0.01,
    gamma=0.99,
    lam=0.95,
    learning_rate=0.0005,
    input_features=8,
    output_features=4,
    hidden_features=128,
    num_episodes=5000,
    running_avg_steps=25,
    log_freq=50,
    device="mps",
    ): 

    # initialize our policy
    policy_network = PolicyNetwork(input_features, output_features, hidden_features).to(device)

    optimizer_policy = optim.Adam(policy_network.parameters(), lr = learning_rate)

    value_network = ValueNetwork(input_features, hidden_features).to(device)

    optimizer_value = optim.Adam(value_network.parameters(), lr = learning_rate)

    log = {
        "scores": [],
        "running_avg_scores": []
    }

    episode_rewards = []

    for i in range(num_episodes): 

        state, _ = env.reset()
        
        log_probs = []
        entropies = []
        rewards = []
        values = []
        done = False
        
        while not done: 

            state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            possible_actions = policy_network(state) # outputs probability of taking said action, shape of (1,4)
            selected_action = torch.multinomial(possible_actions, 1).item() # returns index, action chosen according to output probability
            log_action_probs = torch.log(possible_actions[0, selected_action]) # find the log probability that we selected the action that we selected

            value = value_network(state)
            values.append(value)

            log_probs.append(log_action_probs)

            next_state, reward, terminal, truncated, _ = env.step(selected_action) # you need to pass in an integer not 
            done = terminal or truncated

            rewards.append(reward)

            # compute our entropy = -log(probs of each action) * probs of each action
            entropy = -torch.sum(torch.log(possible_actions+1e-8)*possible_actions)
            entropies.append(entropy)

            state = next_state
        
        values = torch.cat(values, dim=0).squeeze().to(device)
        advantages, returns = compute_gae(rewards, values, gamma, lam)

        advantages = (advantages - advantages.mean())/(advantages.std()+1e-8)

        # compute our loss which is -objective - entropy
        policy_loss = -(torch.sum(torch.stack(log_probs)*advantages)) - entropy_weight * torch.sum(torch.stack(entropies))
        value_loss = F.smooth_l1_loss(values, returns) # use V(s) - G_t, we use G_t which is the value of the action we took from that state as an approximation of the true state value
        # thsi makes sense because after many trajectories, we have sampled most of the actions from that state which ~V(s)

        optimizer_policy.zero_grad()
        optimizer_value.zero_grad()

        policy_loss.backward()
        value_loss.backward()

        torch.nn.utils.clip_grad_norm(policy_network.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1
        torch.nn.utils.clip_grad_norm(value_network.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1
        
        optimizer_policy.step()
        optimizer_value.step()

        total_rewards = np.sum(rewards)

        log["scores"].append(total_rewards)
        running_avg_scores = np.mean(log["scores"][-running_avg_steps:])
        log["running_avg_scores"].append(running_avg_scores)

        if i % log_freq == 0:
            print(f"Episode {i}, Total Reward: {total_rewards}, Average Reward: {running_avg_scores}")

        if running_avg_scores >= 200: 
            print("Training complete")
            break

    return policy, log


In [28]:
### Play Game ###
env = gym.make("LunarLander-v3", render_mode="rgb_array")
policy, log = train(env, device="mps")

/var/folders/jl/ppm07cc55jgfvx0gzqfyhtrw0000gn/T/ipykernel_54907/4113002368.py:83: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(policy_network.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1
/var/folders/jl/ppm07cc55jgfvx0gzqfyhtrw0000gn/T/ipykernel_54907/4113002368.py:84: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(value_network.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1


Episode 0, Total Reward: -141.15516045993044, Average Reward: -141.15516045993044
Episode 50, Total Reward: -118.89907194737845, Average Reward: -168.39978248230946
Episode 100, Total Reward: -112.90950438375764, Average Reward: -131.68114644859284
Episode 150, Total Reward: -100.52245572978121, Average Reward: -204.16453411179552
Episode 200, Total Reward: -69.84160086306431, Average Reward: -196.15675965949347
Episode 250, Total Reward: -1.2634327817085875, Average Reward: -177.542756912846
Episode 300, Total Reward: -22.346919472603076, Average Reward: -146.5802024977357
Episode 350, Total Reward: -177.10842899517849, Average Reward: -139.5449018456971
Episode 400, Total Reward: -80.93820820097154, Average Reward: -184.9194539426104
Episode 450, Total Reward: -9.71450175574974, Average Reward: -99.1896609950739
Episode 500, Total Reward: -265.33443228717994, Average Reward: -80.81415565683864
Episode 550, Total Reward: -59.17090790709217, Average Reward: -18.1151289078683
Episode 60

NameError: name 'policy' is not defined

## Visualize the Trained Agent

Run one episode with the trained policy, capture each frame as an RGB image, and play them back as an animation inside the notebook.

How it works:

1. We make a fresh env with `render_mode="rgb_array"` so each step returns a picture (a NumPy array of pixels).
2. We step through the episode using the policy network — but at inference time we pick the _most likely_ action (`argmax`) instead of sampling, which gives a more deterministic, "best-effort" rollout.
3. We collect the frames into a list, then use `matplotlib.animation.FuncAnimation` to play them back as a video in the notebook.


In [14]:
from IPython.display import HTML

def visualize_agent(policy_network, device="mps", max_steps=1000, deterministic=True, seed=None):
    """Run one episode with the trained policy and return an HTML animation of the rollout."""

    # Fresh env with rgb_array so .render() gives us pixel frames
    vis_env = gym.make("LunarLander-v3", render_mode="rgb_array")
    state, _ = vis_env.reset(seed=seed)

    frames = []
    total_reward = 0.0
    done = False
    steps = 0

    # eval mode disables things like dropout (not used here, but good practice)
    policy_network.eval()

    with torch.no_grad():  # we're not training, so don't track gradients
        while not done and steps < max_steps:
            # capture the current frame (an HxWx3 numpy array of pixels)
            frames.append(vis_env.render())

            # forward pass to get action probabilities
            state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            probs = policy_network(state_t)

            if deterministic:
                # pick the single most-likely action — best-effort rollout
                action = torch.argmax(probs, dim=-1).item()
            else:
                # sample from the policy — shows stochastic behavior
                action = torch.multinomial(probs, 1).item()

            state, reward, terminal, truncated, _ = vis_env.step(action)
            total_reward += reward
            done = terminal or truncated
            steps += 1

    vis_env.close()
    print(f"Episode finished — steps: {steps}, total reward: {total_reward:.2f}")

    # Build the animation. Each frame is a still image; FuncAnimation flips through them.
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis("off")
    img = ax.imshow(frames[0])

    def update(i):
        img.set_data(frames[i])
        return [img]

    # interval is ms between frames. LunarLander runs at ~50 FPS, so 20ms = 50fps playback.
    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=20, blit=True
    )
    plt.close(fig)  # prevent the static first frame from also rendering

    return HTML(anim.to_jshtml())


# Run it. Assumes the training cell returned `policy` (the trained policy network).
visualize_agent(policy, device="mps")

NameError: name 'policy' is not defined